# 🔵 EDA — K-Means: Segmentação de Padrões
## Predictfy × Locaweb — FIAP Challenge 2026

Notebook exploratório: treina K-Means e visualiza clusters, PCA e perfis.

**Input:** `data/processed/incidents_features.parquet`
**Módulo:** `src.models.kmeans_model`


In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.kmeans_model import load_data, train, CLUSTER_FEATURES, TARGET, K_MIN

plt.rcParams.update({
    "figure.figsize": (14, 6),
    "figure.dpi": 100,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("Carregando dados e treinando K-Means...")
df = load_data()
results = train(df)

k_final  = results["k_final"]
df_cl    = results["df_cl"]
X_pca    = results["X_pca"]
pca      = results["pca"]
print(f"K selecionado: {k_final}")

## 1. Perfil Normalizado dos Clusters

In [ ]:
perfil = df_cl.groupby("cluster")[CLUSTER_FEATURES].mean()
perfil_norm = (perfil - perfil.min()) / (perfil.max() - perfil.min() + 1e-9)
perfil_norm.index = [f"Cluster {c}" for c in perfil.index]

fig, ax = plt.subplots(figsize=(14, max(3, k_final + 1)))
sns.heatmap(perfil_norm, annot=True, fmt=".2f", cmap="RdYlGn",
            linewidths=0.5, ax=ax)
ax.set_title("Perfil Normalizado dos Clusters (0=mín · 1=máx)", fontsize=12, fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.show()

## 2. Visualização PCA 2D

In [ ]:
CORES = ["#1a73e8","#ff9f0a","#30d158","#ff2d55","#af52de","#5ac8fa","#ff6b35","#34c759"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for c in sorted(df_cl["cluster"].unique()):
    mask = df_cl["cluster"] == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=CORES[c % len(CORES)], alpha=0.3, s=5,
                    label=f"Cluster {c} (n={mask.sum():,})")
axes[0].set_title("Clusters — PCA 2D", fontsize=12, fontweight="bold")
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
axes[0].legend(fontsize=9, markerscale=3); axes[0].grid(alpha=0.2)

cor_viol = df_cl[TARGET].map({0: "#5ac8fa", 1: "#ff2d55"}).values
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=cor_viol, alpha=0.3, s=5)
axes[1].set_title("Violações OLA — PCA 2D (vermelho=SIM)", fontsize=12, fontweight="bold")
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
axes[1].grid(alpha=0.2)

plt.tight_layout(); plt.show()

## 3. Taxa de Violação por Cluster

In [ ]:
cluster_info = results["cluster_info"]
CORES = ["#1a73e8","#ff9f0a","#30d158","#ff2d55","#af52de","#5ac8fa","#ff6b35","#34c759"]

ids    = [c["id"] for c in cluster_info]
taxas  = [c["taxa_violacao_pct"] for c in cluster_info]
sizes_ = [c["tamanho"] for c in cluster_info]
nomes  = [c["nome"] for c in cluster_info]
cores_ = [CORES[c["id"] % len(CORES)] for c in cluster_info]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars0 = axes[0].bar([f"C{i}" for i in ids], taxas, color=cores_, edgecolor="white")
axes[0].set_title("Taxa de Violação por Cluster (%)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("% Violações")
for bar, v in zip(bars0, taxas):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.02, f"{v:.2f}%", ha="center", va="bottom")
axes[0].grid(axis="y", alpha=0.3)

bars1 = axes[1].bar([f"C{i}" for i in ids], sizes_, color=cores_, edgecolor="white")
axes[1].set_title("Tamanho dos Clusters", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Nº de incidentes")
for bar, v in zip(bars1, sizes_):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 50, f"{v:,}", ha="center", va="bottom", fontsize=9)
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout(); plt.show()

print("Perfil de cada cluster:")
for c in cluster_info:
    print(f"  Cluster {c['id']}: {c['nome']} | {c['tamanho']:,} inc. | {c['taxa_violacao_pct']:.2f}% violações")